# 🧱 Deep Learning Project: Cracked vs. Non-Cracked Surface Classification
### Course: Neural Networks (ELE / AI Program) - NINU
**Author**: Ammar Abdelfattah & Team  
**Deadline**: December 12, 2026  
**Target**: Binary Surface Defect Classification (Cracked vs. Non-Cracked)

---

## 📌 Project Overview & Requirements
The goal is to build, train, evaluate, and compare **4 distinct neural network architectures**:
1. **FFNN (Feedforward Neural Network / Multi-Layer Perceptron)**
2. **RNN (Recurrent Neural Network with LSTM cells for spatial-sequence modeling)**
3. **Custom Deep CNN (Convolutional Neural Network)**
4. **Pretrained CNN Transfer Learning (MobileNetV2 / ResNet50)**

### Hyperparameters to Systematically Explore:
- **Batch size**: e.g., 16 vs 32 vs 64
- **Number of layers / Hidden capacity**: Shallow vs Deep
- **Dropout Rate**: 0.0 vs 0.2 vs 0.5 (Overfitting defense)
- **Optimizers**: Adam vs SGD (with Momentum) vs RMSprop
- **Regularization (Weight Decay / L2)**: No reg vs $10^{-4}$ vs $10^{-3}$
- **Learning Rate & Schedulers**: Fixed vs Exponential / ReduceLROnPlateau


In [ ]:
# 1. Environment Setup and Library Imports
import os
import sys
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# TensorFlow & Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, optimizers, callbacks

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


---
## 📥 Part I: Data Preparation & Exploration

Dataset: [Kaggle Cracked vs Non-cracked surface datasets](https://www.kaggle.com/datasets/geadalfa/cracked-non-cracked-surface-datasets/data)  
Each image belongs to one of two categories:
- `Positive` / `Cracked`: Surface containing concrete cracks.
- `Negative` / `Non-cracked`: Clean surface without structural anomalies.


In [ ]:
# 2. Dataset Acquisition Helper
# You can download via kagglehub / opendatasets, or specify local folder path
DATA_DIR = "./dataset"

def setup_dataset():
    if not os.path.exists(DATA_DIR):
        os.makedirs(os.path.join(DATA_DIR, "Positive"), exist_ok=True)
        os.makedirs(os.path.join(DATA_DIR, "Negative"), exist_ok=True)
        print("Dataset directory prepared. Please place 'Positive' and 'Negative' folders inside './dataset'")
        print("For rapid local verification, generating synthetic crack/non-crack samples...")
        # Synthetic mock generator for standalone offline verification
        for i in range(50):
            # Negative: Smooth surface with subtle Gaussian noise
            neg_img = (np.random.normal(128, 15, (128, 128)) + 50).clip(0, 255).astype(np.uint8)
            plt.imsave(os.path.join(DATA_DIR, "Negative", f"clean_{i:03d}.jpg"), neg_img, cmap='gray')
            
            # Positive: Surface with jagged crack lines
            pos_img = (np.random.normal(128, 15, (128, 128)) + 50).clip(0, 255).astype(np.uint8)
            x = np.arange(10, 118)
            y = (64 + 20 * np.sin(x / 10) + np.random.normal(0, 3, len(x))).astype(int).clip(0, 127)
            for xi, yi in zip(x, y):
                pos_img[max(0, yi-1):min(128, yi+2), max(0, xi-1):min(128, xi+2)] = 10
            plt.imsave(os.path.join(DATA_DIR, "Positive", f"crack_{i:03d}.jpg"), pos_img, cmap='gray')
        print("Generated 50 sample images per class for immediate notebook execution.")
    else:
        print(f"Dataset found at {DATA_DIR}")

setup_dataset()


In [ ]:
# 3. Data Loading and Visual Inspection
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

def load_image_dataset(data_dir, img_size=(128, 128), batch_size=32):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=img_size,
        batch_size=batch_size,
        label_mode="binary"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=img_size,
        batch_size=batch_size,
        label_mode="binary"
    )
    return train_ds, val_ds

train_ds, val_ds = load_image_dataset(DATA_DIR, IMG_SIZE, BATCH_SIZE)
class_names = train_ds.class_names
print("Classes detected:", class_names)


In [ ]:
# 4. Data Visualization
plt.figure(figsize=(10, 5))
for images, labels in train_ds.take(1):
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"Label: {class_names[int(labels[i])]}")
        plt.axis("off")
plt.tight_layout()
plt.show()


---
## ⚙️ Shared Training and Evaluation Utilities
We define modular `train_model` and `evaluate_model` functions that log:
- Training and Validation Loss / Accuracy
- Epoch execution time
- Final Test Metrics (Confusion Matrix, Precision, Recall, F1)


In [ ]:
# 5. Core Training & Evaluation Routines
def train_model(model, train_data, val_data, epochs=15, callbacks_list=None):
    start_time = time.time()
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks_list,
        verbose=1
    )
    elapsed = time.time() - start_time
    print(f"⏱️ Training finished in {elapsed:.2f} seconds.")
    return history, elapsed

def evaluate_model(model, val_data, model_name="Model"):
    loss, acc = model.evaluate(val_data, verbose=0)
    print(f"📊 {model_name} Results -> Loss: {loss:.4f} | Accuracy: {acc*100:.2f}%")
    
    y_true, y_pred = [], []
    for imgs, lbls in val_data:
        preds = model.predict(imgs, verbose=0)
        y_true.extend(lbls.numpy())
        y_pred.extend((preds > 0.5).astype(int))
    
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    cm = confusion_matrix(y_true, y_pred)
    print("
Classification Report:
", classification_report(y_true, y_pred, target_names=["Negative", "Positive"]))
    
    # Plot Confusion Matrix
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
    plt.title(f"{model_name} Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()
    
    return {"name": model_name, "loss": loss, "accuracy": acc, "cm": cm}

def plot_history(history, title="Training History"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train Acc', marker='o')
    ax1.plot(history.history['val_accuracy'], label='Val Acc', marker='s')
    ax1.set_title(f"{title} - Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    # Loss
    ax2.plot(history.history['loss'], label='Train Loss', marker='o')
    ax2.plot(history.history['val_loss'], label='Val Loss', marker='s')
    ax2.set_title(f"{title} - Loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.show()


---
## 🧠 Model 1: Feedforward Neural Network (FFNN / MLP)
Flattens the 2D image ($128 \times 128 \times 3$) into a single 1D vector ($49,152$ features) and passes it through fully connected layers with batch normalization and dropout.


In [ ]:
# 6. Model 1: FFNN Architecture
def build_ffnn(input_shape=(128, 128, 3), hidden_units=[256, 128, 64], dropout_rate=0.3, l2_reg=1e-4, lr=1e-3, optimizer_name="adam"):
    model = keras.Sequential([
        layers.Rescaling(1./255, input_shape=input_shape),
        layers.Flatten()
    ])
    
    for units in hidden_units:
        model.add(layers.Dense(units, kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
            
    model.add(layers.Dense(1, activation='sigmoid'))
    
    opt = {
        'adam': optimizers.Adam(learning_rate=lr),
        'sgd': optimizers.SGD(learning_rate=lr, momentum=0.9),
        'rmsprop': optimizers.RMSprop(learning_rate=lr)
    }.get(optimizer_name.lower(), optimizers.Adam(learning_rate=lr))
    
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_ffnn = build_ffnn()
model_ffnn.summary()

history_ffnn, t_ffnn = train_model(model_ffnn, train_ds, val_ds, epochs=10)
plot_history(history_ffnn, "FFNN")
res_ffnn = evaluate_model(model_ffnn, val_ds, "FFNN")


---
## 🔄 Model 2: Recurrent Neural Network (LSTM for Spatial Rows)
An image is treated as a sequence of spatial scan lines. An image of shape $(128, 128, 3)$ can be interpreted as a sequence of $128$ time steps, each containing $128 \times 3 = 384$ spatial features.


In [ ]:
# 7. Model 2: RNN (LSTM) Architecture
def build_rnn_lstm(input_shape=(128, 128, 3), lstm_units=64, dropout_rate=0.2, lr=1e-3):
    model = keras.Sequential([
        layers.Rescaling(1./255, input_shape=input_shape),
        # Reshape (128, 128, 3) -> (128, 384) sequence
        layers.Reshape((input_shape[0], input_shape[1] * input_shape[2])),
        layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=False)),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_rnn = build_rnn_lstm()
model_rnn.summary()

history_rnn, t_rnn = train_model(model_rnn, train_ds, val_ds, epochs=10)
plot_history(history_rnn, "RNN (LSTM)")
res_rnn = evaluate_model(model_rnn, val_ds, "RNN-LSTM")


---
## 🖼️ Model 3: Custom Deep Convolutional Neural Network (CNN)
A hierarchical convolutional network preserving spatial 2D relationships using $3 \times 3$ receptive fields, MaxPooling, BatchNormalization, and Dropout.


In [ ]:
# 8. Model 3: Custom CNN Architecture
def build_custom_cnn(input_shape=(128, 128, 3), num_blocks=3, dropout_rate=0.3, l2_reg=1e-4, lr=1e-3, optimizer_name="adam"):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)
    
    filters = 32
    for i in range(num_blocks):
        x = layers.Conv2D(filters, (3, 3), padding='same', kernel_regularizer=regularizers.l2(l2_reg))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D((2, 2))(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate * 0.5)(x)
        filters *= 2
        
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs)
    
    opt = {
        'adam': optimizers.Adam(learning_rate=lr),
        'sgd': optimizers.SGD(learning_rate=lr, momentum=0.9),
        'rmsprop': optimizers.RMSprop(learning_rate=lr)
    }.get(optimizer_name.lower(), optimizers.Adam(learning_rate=lr))
    
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_cnn = build_custom_cnn(num_blocks=3)
model_cnn.summary()

history_cnn, t_cnn = train_model(model_cnn, train_ds, val_ds, epochs=10)
plot_history(history_cnn, "Custom CNN")
res_cnn = evaluate_model(model_cnn, val_ds, "Custom CNN")


---
## 🚀 Model 4: Transfer Learning with Pretrained CNN (MobileNetV2)
Leverages deep representations learned on millions of ImageNet images. The pretrained base acts as a feature extractor, followed by fine-tuning.


In [ ]:
# 9. Model 4: Pretrained Transfer Learning Architecture
def build_pretrained_model(input_shape=(128, 128, 3), fine_tune_at=100, lr=1e-4):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    # Freeze base initially
    base_model.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model, base_model

model_pretrained, base_mobilenet = build_pretrained_model()
model_pretrained.summary()

history_pre, t_pre = train_model(model_pretrained, train_ds, val_ds, epochs=8)
plot_history(history_pre, "Pretrained MobileNetV2 (Feature Extraction)")
res_pre = evaluate_model(model_pretrained, val_ds, "Pretrained MobileNetV2")


---
## 🔬 Hyperparameter Exploration & Sensitivity Analysis
As mandated by the project guidelines, we systematically explore:
1. **Optimizers**: Adam vs SGD vs RMSprop
2. **Dropout Rate**: 0.0, 0.3, 0.5
3. **Weight Decay (L2)**: 0.0, 1e-4, 1e-2
4. **Learning Rate Schedulers**: Exponential Decay vs ReduceLROnPlateau


In [ ]:
# 10. Systematic Optimizer Comparison on Custom CNN
optimizers_to_test = ["adam", "sgd", "rmsprop"]
optimizer_results = {}

for opt_name in optimizers_to_test:
    print(f"\n🧪 Testing Optimizer: {opt_name.upper()}...")
    m = build_custom_cnn(optimizer_name=opt_name, lr=1e-3)
    hist, el = train_model(m, train_ds, val_ds, epochs=6)
    val_acc = max(hist.history['val_accuracy'])
    optimizer_results[opt_name] = val_acc
    print(f"Result for {opt_name}: Best Val Accuracy = {val_acc*100:.2f}%")

plt.figure(figsize=(6, 3))
plt.bar(optimizer_results.keys(), [v*100 for v in optimizer_results.values()], color=['#4C72B0', '#55A868', '#C44E52'])
plt.ylabel("Validation Accuracy (%)")
plt.title("Comparison of Optimizers")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# 11. Learning Rate Scheduler Experiment
lr_schedule = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
)

model_sched = build_custom_cnn(lr=1e-3)
hist_sched, _ = train_model(model_sched, train_ds, val_ds, epochs=8, callbacks_list=[lr_schedule])
plot_history(hist_sched, "CNN with ReduceLROnPlateau")


---
## 🏆 Final Benchmark & Model Comparison


In [ ]:
# 12. Model Benchmark Summary Table
results = [res_ffnn, res_rnn, res_cnn, res_pre]
df_results = pd.DataFrame([
    {
        "Model Architecture": r["name"],
        "Validation Accuracy (%)": f"{r['accuracy']*100:.2f}%",
        "Loss": f"{r['loss']:.4f}",
    }
    for r in results
])

print("\n" + "="*50)
print("       FINAL ARCHITECTURE COMPARISON TABLE")
print("="*50)
print(df_results.to_string(index=False))

# Plot Comparison Bar Chart
plt.figure(figsize=(8, 4))
plt.bar(df_results["Model Architecture"], [float(x.replace('%','')) for x in df_results["Validation Accuracy (%)"]], color='teal')
plt.ylabel("Accuracy (%)")
plt.ylim(0, 105)
plt.title("Comparison of Model Accuracies on Surface Crack Dataset")
for i, v in enumerate([float(x.replace('%','')) for x in df_results["Validation Accuracy (%)"]]):
    plt.text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


---
## 📝 Conclusion and Recommendations
1. **FFNN** struggles with spatial shift invariance and requires excessive parameters when flattened, leading to slow convergence and overfitting.
2. **RNN (LSTM)** can model sequential row correlations, but is computationally heavy for 2D image signals.
3. **Custom CNN** provides strong inductive bias through 2D weight sharing and spatial downsampling, achieving excellent trade-off between speed and accuracy.
4. **Pretrained CNN (Transfer Learning)** delivers superior generalization, especially with small datasets, due to rich pre-existing feature representations from ImageNet.
